In [1]:
from chaining import get_chain
from ragas import evaluate
from chaining import get_chain, get_retriever
from verification_datas import datas

In [3]:
chain = get_chain()
retriever = get_retriever()

questions = [x["question"] for x in datas]
targets = [x["answer"] for x in datas]


In [ ]:
from pprint import pprint as pp
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

target = questions[0]
pp(target)

llm = ChatOpenAI(model_name="gpt-5", api_key=os.getenv("OPENAI_API_KEY"))
prompt = ChatPromptTemplate.from_template(
    """
        당신은 **형사법** 법령 검색 도움 봇입니다. 
        제공된 {context}를 바탕으로 사용자의 {question}에 답변해 주세요.
        Question이 **형사법**에 해당하지 않는 법령에 대한 질문이라면 관련 법령을 검색할 수 없다고 답변하고 전문가의 도움을 받기를 제안하세요.

        **Required Rule**
            1. 반드시 답변의 근거가 된 참고자료({context})를 포함해야 합니다.
            2. {context}가 없을 경우 관련 법령을 검색할 수 없다고 답변하고 전문가의 도움을 받기를 제안하세요.
            3. **형사법**에 대한 답변만을 생성해야 합니다.
            4. 당신은 법령 정보만 제공합니다. 추가 의견은 추가하지 않습니다.

        Context:
        {context}

        Question:
        {question}
    """
)

def format_str(docs: list) -> str:
    return "\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


('A: 피해자, B: 주도자/ A의 집 비밀번호, 금괴가 든 금고위치, 금고 비밀번도, 도주로 등 정보와 범행계획 제공, C: A의 집에 '
 '침입, 절도 시도 했으나 금고안에 금괴는 없었으며 A에게 발각되어 A를 폭행 후 도주, D: C가 범행을 하는 동안 A의 집앞에서 망을 '
 '보다 자리 이탈. A는 B,C,D의 처벌을 원하는 상태야. B,C,D가 받을 수 있는 처벌은 어떤게 있는지 알려줘.')


In [12]:
docs = retriever.invoke(target)
pp(docs)

[Document(metadata={'file_name': '경범죄 처벌법', 'url': 'https://www.law.go.kr/LSW/lsInfoP.do?lsiSeq=198226&ancYd=&ancNo=&efYd=20171024&nwJoYnInfo=Y&ancYnChk=0&efGubun=Y&vSct=*#0000', 'title': '경범죄 처벌법', 'seq': 2, '_id': '1bc0dded-0d5e-41e9-a8eb-1dce6163bca8', '_collection_name': 'criminal'}, page_content='경범죄 처벌법\n제3조(경범죄의 종류)① 다음 각 호의 어느 하나에 해당하는 사람은 10만원 이하의 벌금, 구류 또는 과료(科料)의 형으로 처벌한다.<개정 2014. 11. 19., 2017. 7. 26., 2017. 10. 24.>1. (빈집 등에의 침입) 다른 사람이 살지 아니하고 관리하지 아니하는 집 또는 그 울타리ㆍ건조물(建造物)ㆍ배ㆍ자동차 안에 정당한 이유 없이 들어간 사람2. (흉기의 은닉휴대) 칼ㆍ쇠몽둥이ㆍ쇠톱 등 사람의 생명 또는 신체에 중대한 위해를 끼치거나 집이나 그 밖의 건조물에 침입하는 데에 사용될 수 있는 연장이나 기구를 정당한 이유 없이 숨겨서 지니고 다니는 사람3. (폭행 등 예비) 다른 사람의 신체에 위해를 끼칠 것을 공모(共謀)하여 예비행위를 한 사람이 있는 경우 그 공모를 한 사람4. 삭제<2013. 5. 22.>5. (시체 현장변경 등) 사산아(死産兒)를 감추거나 정당한 이유 없이 변사체 또는 사산아가 있는 현장을 바꾸어 놓은 사람6. (도움이 필요한 사람 등의 신고불이행) 자기가 관리하고 있는 곳에 도움을 받아야 할 노인, 어린이, 장애인, 다친 사람 또는 병든 사람이 있거나 시체 또는 사산아가 있는 것을 알면서 이를 관계 공무원에게 지체 없이 신고하지 아니한 사람7. (관명사칭 등) 국내외의 공직(公職), 계급, 훈장, 학위 또는 그 밖에 법령에 따라 정하여진 명칭이나 칭호 등을 거짓으로 꾸

In [36]:
import time
from tqdm import tqdm

dataset=[]
for i, query in tqdm(enumerate(questions), 'make dataset'):
    response = chain.invoke(query)
    docs = retriever.invoke(query)
    contexts = [doc.page_content for doc in docs]

    data = {
        'response': response,
        'retrieved_contexts': contexts,
        'user_input': query,
        'reference': targets[i],
    }
    dataset.append(data)

print(dataset)

make dataset: 6it [07:23, 73.88s/it] 

[{'response': '요청하신 사안은 형법상 주거침입, 절도(미수), 체포면탈을 위한 폭행·협박(준강도) 또는 폭행·상해, 공범 성립(공동정범·교사·종범) 등이 핵심입니다. 그러나 현재 제공된 법령(경범죄처벌법 제3조, 특정범죄가중처벌법 제5조의9, 국가보안법 제4조)만으로는 B·C·D의 처벌 규정을 특정할 수 없습니다. 위 세 법령은 본 사안(사적 주거 침입·절도·폭행·공범)에 직접 적용되지 않습니다. 따라서 관련 형법 조항에 대한 검토가 필요하나, 현 단계에서 추가 형법 조문을 제공할 수 없어 관련 법령을 검색할 수 없습니다. 전문가의 도움을 받으시기 바랍니다.\n\n참고자료(제공됨)\n경범죄 처벌법\n제3조(경범죄의 종류)① 다음 각 호의 어느 하나에 해당하는 사람은 10만원 이하의 벌금, 구류 또는 과료(科料)의 형으로 처벌한다.<개정 2014. 11. 19., 2017. 7. 26., 2017. 10. 24.>1. (빈집 등에의 침입) 다른 사람이 살지 아니하고 관리하지 아니하는 집 또는 그 울타리ㆍ건조물(建造物)ㆍ배ㆍ자동차 안에 정당한 이유 없이 들어간 사람2. (흉기의 은닉휴대) 칼ㆍ쇠몽둥이ㆍ쇠톱 등 사람의 생명 또는 신체에 중대한 위해를 끼치거나 집이나 그 밖의 건조물에 침입하는 데에 사용될 수 있는 연장이나 기구를 정당한 이유 없이 숨겨서 지니고 다니는 사람3. (폭행 등 예비) 다른 사람의 신체에 위해를 끼칠 것을 공모(共謀)하여 예비행위를 한 사람이 있는 경우 그 공모를 한 사람4. 삭제<2013. 5. 22.>5. (시체 현장변경 등) 사산아(死産兒)를 감추거나 정당한 이유 없이 변사체 또는 사산아가 있는 현장을 바꾸어 놓은 사람6. (도움이 필요한 사람 등의 신고불이행) 자기가 관리하고 있는 곳에 도움을 받아야 할 노인, 어린이, 장애인, 다친 사람 또는 병든 사람이 있거나 시체 또는 사산아가 있는 것을 알면서 이를 관계 공무원에게 지체 없이 신고하지 아니한 사람7. (관명사칭 등) 국내외의 공직(公職), 계급, 훈장, 학위 또는 

In [4]:
dataset

{'reference': ['B,C,D는 모두 폭처법상 공동주거침입죄와 강도치상죄의 죄책을 지고, B,D는 C가 A를 폭행하고 상해가 발생할 것을 예견할 수 없었다면 상해 결과에 대하여는 아무런 죄책을 지지 않고 합동 특수절도죄의 미수범 죄책만을 진다.',
  '위법성 인식을 고의와 독립된 책임요소로 보는 견해에 의하면 오인에 정당한 이유가 있으면 질문자님이 상대에게 상해를 입힌 행위는 상해의 고의가 인정되나 책임이 조각됩니다. 참고 법 조문 : 형법 제16조(법률의 착오) 자기의 행위가 법령에 의하여 죄가 되지 아니하는 것으로 오인한 행위는 그 오인에 정당한 이유가 있는 때에 한하여 벌하지 아니한다.',
  '협박죄에 있어 해악을 가하는 것을 고지하는 행위는 통상 언어에 의한 것이나 경우에 따라서는 한마디 말도 없이 거동에 의해서 고지할 수 있으므로 신체에 대해 위해를 가할 고지로 볼 수 있어 협박죄가 성립합니다. 참고 법 조문 : 형법 제283조(협박)제1항  사람을 협박한 자는 3년 이하의 징역, 500만원 이하의 벌금, 구류 또는 과료에 처한다.',
  '책임을 피할 수 없습니다. 피해자에게 심장질환이 있었거나 만취 상태였다는 사실이 사망에 기여했더라도, 가해자의 폭행이 사망의 직접적인 계기가 되었다면 인과관계는 단절되지 않습니다. 참고 법 조문 : 형법 제15조(특별히 중한 죄되지 아니하는 경우)제2항  결과로 인하여 형이 중할 죄에 있어서 그 결과의 발생을 예견할 수 없었을 때에는 중한 죄로 벌하지 아니한다.',
  '재미삼아 지갑 훔쳤어도 불능미수로 절도죄(미수 포함)로 처벌받을 가능성이 매우 높습니다. 참고 법 조문 : 형법 제 27조 (불능범) 실행의 수단 또는 대상의 착오로 인하여 결과의 발생이 불가능하더라도 위험성이 있는 때에는 처벌한다. 형법 제329조(절도죄의 성립) (절도) 타인의 재물을 절취한 자는 6년 이하의 징역 또는 1천만원 이하의 벌금에 처한다. 형법 제 342조 (미수범) 제329조 내지 제341조의 미수범은 처벌한다.',


In [59]:
import time
from ragas import experiment
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness, AnswerRelevancy
from pydantic import BaseModel
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from openai import AsyncOpenAI
from pprint import pprint
from tqdm import tqdm

client = AsyncOpenAI(
    max_retries=5,   # 재시도 횟수 상향
    timeout=120,     # 타임아웃 연장
)
eval_llm = llm_factory(model="gpt-4o", client=client)
eval_embeddings = embedding_factory(provider="openai", model="text-embedding-3-large", client=client)

class DatasetRow(BaseModel):
    response: str
    retrieved_contexts: list[str]
    user_input: str
    reference: str

class ExperimentResult(BaseModel):
    faithfulness: float
    context_recall: float
    answer_relevancy: float
    context_precision: float

faithfulness = Faithfulness(llm=eval_llm)
context_recall = ContextRecall(llm=eval_llm)
answer_relevancy = AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)
context_precision = ContextPrecision(llm=eval_llm)

@experiment(ExperimentResult)
async def run_evaluation(row):
    try:
        faithfulness_result = await faithfulness.ascore(
            user_input=row.user_input,
            response=row.response,
            retrieved_contexts=row.retrieved_contexts
        )

        relevancy_result = await answer_relevancy.ascore(
            user_input=row.user_input,
            response=row.response
        )
        
        context_recall_result = await context_recall.ascore(
            user_input=row.user_input,
            retrieved_contexts=row.retrieved_contexts,
            reference=row.reference
        )
        
        context_precision_result = await context_precision.ascore(
            user_input=row.user_input,
            reference=row.reference,
            retrieved_contexts=row.retrieved_contexts
        )

        return ExperimentResult(
            faithfulness=faithfulness_result.value,
            context_recall=context_recall_result.value,
            answer_relevancy=relevancy_result.value,
            context_precision=context_precision_result.value
        )
    except Exception as e:
        print('error eval', e)
        return ExperimentResult(
            faithfulness=0,
            context_recall=0,
            answer_relevancy=0,
            context_precision=0
        )


result = []
for row in tqdm(dataset, desc="Evaluating"):
    data = DatasetRow.model_validate(row)
    score = await run_evaluation(data)
    result.append(score)
    print(score)
    time.sleep(20)

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

faithfulness=0.42857142857142855 context_recall=0.0 answer_relevancy=0.0 context_precision=0.0


Evaluating:  17%|█▋        | 1/6 [01:02<05:10, 62.02s/it]

faithfulness=0.75 context_recall=0.0 answer_relevancy=0.0 context_precision=0.0


Evaluating:  33%|███▎      | 2/6 [01:55<03:48, 57.16s/it]

...

Evaluating:  67%|██████▋   | 4/6 [04:33<02:27, 73.57s/it]

faithfulness=0.6666666666666666 context_recall=0.25 answer_relevancy=0.5361540300586255 context_precision=0.0


Evaluating:  83%|████████▎ | 5/6 [05:34<01:09, 69.06s/it]

faithfulness=0.6666666666666666 context_recall=0.0 answer_relevancy=0.0 context_precision=0.0


Evaluating: 100%|██████████| 6/6 [06:40<00:00, 66.83s/it]


In [ ]:
result

ExperimentResult(faithfulness=0.6666666666666666, context_recall=0.0, answer_relevancy=0.0, context_precision=0.0)